# EMNIST Cloud Run Inference Service — Setup and Validation

This notebook documents the **EMNIST Balanced (47-class) inference service hosted on Google Cloud Run** and the deployment evidence preserved in this branch. The service is used by the cloud-assisted FPGA experiments, but its development and diagnostic testing are recorded separately from the FPGA measurement results.

**Service:** `emnist-cloud-inference`  
**Region:** `us-east1`  
**Service URL:** `https://emnist-cloud-inference-221036165238.us-east1.run.app`  
**Inference route:** `/predict`  
**Input:** 784 raw image bytes (`application/octet-stream`)  
**Response:** JSON containing an integer `prediction` in the EMNIST Balanced class range (0–46).

The code, model checkpoint, and container setup are archived in [`EMNIST_Cloud_Run_Service.zip`](EMNIST_Cloud_Run_Service.zip).

## 1. Relationship to the MNIST implementation

The EMNIST service reused the **MNIST Cloud Run deployment workflow**, including a Python inference application, container packaging, an HTTPS prediction endpoint, and test requests from Cloud Shell. The primary adaptation was replacing the ten-class MNIST model with the trained **47-class EMNIST Balanced CNN** and loading its corresponding checkpoint. Because the dataset still uses 28 × 28 grayscale images, the existing ZedBoard raw-byte request format was preserved.

This notebook concentrates on the changes and evidence for EMNIST rather than repeating the full MNIST cloud service setup.

## 2. Archived service and diagnostic files

| Repository file | Role |
|---|---|
| [`EMNIST_Cloud_Run_Service.zip`](EMNIST_Cloud_Run_Service.zip) | Service source and model package used for deployment |
| [`EMNIST_Cloud_Test_Batch_0900.zip`](EMNIST_Cloud_Test_Batch_0900.zip) | Diagnostic test package for EMNIST images 900–999 |
| [`emnist_cloud_service.png`](emnist_cloud_service.png) | Cloud Shell deployment and service response |
| [`emnist_cloud_service_tes100.png`](emnist_cloud_service_tes100.png) | Diagnostic batch-test results |
| [`emnist_cloud_local_test1.png`](emnist_cloud_local_test1.png) | Direct `/predict` request and JSON response |
| [`emnist_cloud_local_test5.png`](emnist_cloud_local_test5.png) | Additional Cloud Shell classification verification |

The source archive shown in Cloud Run includes `main.py`, `Dockerfile`, `requirements.txt`, `README.md`, and `best_cnn_checkpoint.pt`. These are **archive contents**, not necessarily individual files at repository root. The checkpoint filename comes from the Cloud Run Source screenshot.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

archives = [Path("EMNIST_Cloud_Run_Service.zip"), Path("EMNIST_Cloud_Test_Batch_0900.zip")]
for archive in archives:
    print(f"\n{archive.name}")
    if not archive.is_file():
        print("  Archive not found in working directory. Clone or open the branch to inspect it.")
        continue
    with ZipFile(archive) as zf:
        for item in zf.namelist():
            print(" ", item)

## 3. Model and inference service

The Cloud Run **Source** view shows a Python Flask application importing NumPy and PyTorch. The deployed `CompactCNN` defines three convolutional stages with 16, 32, and 64 channels, each using ReLU activations, with pooling after the first two stages. Global adaptive average pooling feeds a linear classifier with **47 outputs**. The service loads `best_cnn_checkpoint.pt` and runs inference on the CPU.

The source was configured to accept the **same 784-byte raw grayscale input** used by the ZedBoard client and to return the classification as a JSON value. The service implementation is preserved in the ZIP archive, so this notebook does not reproduce the full application source.

![Cloud Run source files and EMNIST CNN definition](emnist_cloud_run_source.png)

*Figure: Deployed source listing and the beginning of the EMNIST CNN implementation in Cloud Run.*

## 4. Container deployment and service verification

The development record shows deployment from Cloud Shell using a Dockerfile-based Cloud Run build in `us-east1`. The service was named `emnist-cloud-inference`, and the deployment output identified a serving revision and service URL. The root endpoint subsequently responded with a message confirming that the EMNIST Balanced compact-CNN service was running.

![Cloud Run deployment and service response](emnist_cloud_service.png)

A separate direct POST test sent **784 zero-valued bytes** with `Content-Type: application/octet-stream` to `/predict`. The recorded service response was HTTP 200 with `{"prediction":5}`. This is a **request/response integration check**, not a correct-label accuracy test.

![Direct binary image request and returned prediction](emnist_cloud_local_test1.png)

### Deployment command recorded during development

The Cloud Shell deployment followed this command structure:

```bash
gcloud run deploy emnist-cloud-inference \
  --source . \
  --region us-east1 \
  --allow-unauthenticated
```

The command is retained as **historical documentation**, not a recommendation to redeploy the service or enable public access. The currently displayed security configuration must be assessed separately (Section 6).

## 5. Diagnostic classification testing

The EMNIST test package was used to check the deployed service against labeled images. The evidence includes a terminal comparison of expected and predicted classes for images 900–999 and a recorded result of **91 correct predictions out of 100 (91.0%)**.

![EMNIST batch diagnostic output](emnist_cloud_service_tes100.png)

![Additional EMNIST classification checks](emnist_cloud_local_test5.png)

**Measurement boundary:** The 91/100 result was a **direct Cloud Run diagnostic** and was explicitly labeled *not a formal FPGA cloud-accuracy measurement*. Formal FPGA accuracy, latency, and throughput are calculated from the ZedBoard SDK records in the separate **EMNIST FPGA Cloud Classification Measurement Notebook and Data** branch. The two sets of observations should not be combined.

## 6. Networking and security configuration

The networking screenshot documents an enabled HTTPS service endpoint and **Ingress: All**, allowing direct internet-originated traffic to reach the service endpoint. End-to-end HTTP/2 and session affinity were not selected in the captured view.

![Cloud Run networking settings](emnist_cloud_run_networking.png)

The security screenshot shows **Require authentication** selected in the interface, but it also displays a warning that the service **remained publicly accessible because `allUsers` had been granted permission**. Therefore, the selected radio option alone did **not** establish effective authenticated-only access. The screenshot also shows a Google-managed encryption key, while Cloud Run threat detection was disabled.

![Cloud Run security settings and public access warning](emnist_cloud_run_security.png)

These are **observed configuration states at the time of the screenshots**. They are not evidence that access restrictions or security protections were subsequently changed. The embedded ZedBoard client's separate TLS certificate-verification and entropy limitations are addressed in the dissertation security evaluation.

## 7. Cloud Run observability evidence

Cloud Run metrics show request activity, request latency, instance activity, CPU utilization, and memory utilization during the experimental period. These service-side observations provide context for the remote inference workload but do not replace the FPGA client's request-response measurements.

![Request counts and latency plots](emnist_cloud_run_metrics1.png)

![Instances, CPU, and memory plots](emnist_cloud_run_metrics2.png)

The graphs contain visibly longer latency observations. **The screenshots do not, by themselves, establish which requests involved cold starts**, nor do they provide a machine-readable series for computing precise averages. Formal cloud-assisted FPGA latency results are derived from its SDK records, not estimates read off these graphs.

## 8. Cloud Run versus FPGA measurement responsibilities

| Component | Responsibility |
|---|---|
| ZedBoard programmable logic | Camera acquisition, downsampling, image storage, and result display |
| Zynq ARM client | Mailbox transfer, HTTPS requests, and response handling |
| Google Cloud Run | Load EMNIST CNN, execute remote classification, and return the prediction |
| This branch | Preserve service source archive, deployment evidence, direct diagnostics, and Cloud Run configuration |
| FPGA cloud measurement branches | Preserve controlled 100-image blocks, formal SDK data, latency/throughput analysis, power testing, and Vivado reports |

The endpoint uses the same raw 28 × 28 grayscale image length as MNIST, but the remote model and returned classification range are specific to EMNIST Balanced.

## 9. Scope and reproducibility notes

The service archive preserves the deployable application and trained checkpoint; the test archive and screenshots preserve selected independent checks of deployment and classification behavior. Reproduction requires an appropriate Google Cloud project, current Cloud Run permissions, and the documented container dependencies. No secrets, credentials, or service-account keys should be included in this repository.

The notebook's code cell **only lists archive contents**; it does not contact the deployed endpoint or change Cloud Run resources. The screenshots document settings and observations from the recorded session, not necessarily the service's present configuration.